# Campo Eléctrico

El **Campo Eléctrico** ($\vec{E}$) se define como la fuerza electrostática por unidad de carga. Es una magnitud vectorial que describe la perturbación que una carga eléctrica provoca en el espacio que la rodea.

Se define mediante la siguiente expresión:

$$\vec{E} = \frac{\vec{F}}{q}$$

> **Nota sobre la dirección:** Se considera que la carga de prueba es positiva ($q > 0$). Por lo tanto, el vector de fuerza ($\vec{F}$) siempre apuntará en la misma dirección que el vector de campo eléctrico ($\vec{E}$).

---

## Formulación para una Carga Puntual

Para determinar el campo en un punto $P$ ubicado en la posición $\vec{r}$, producido por una carga $q'$ situada en $\vec{r}'$, utilizamos la diferencia vectorial:

$$\vec{E} = \frac{1}{4\pi\epsilon_{0}} \frac{q'}{|\vec{r} - \vec{r}\,'|^{2}} \frac{(\vec{r} - \vec{r}\,')}{|\vec{r} - \vec{r}\,'|}$$


```{image} images_tikz/01_06.svg
:alt: Campo eléctrico
:width: 60%
:align: center


Simplificando la expresión, obtenemos la forma estándar para el campo eléctrico en el punto $P$:

$$\vec{E} = \frac{1}{4\pi\epsilon_{0}} q' \frac{(\vec{r} - \vec{r}\,')}{|\vec{r} - \vec{r}\,'|^{3}}$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, RadioButtons

def interactuar_campo(q_val, x_c, y_c):
    # 1. Crear la rejilla
    x = np.linspace(-5, 5, 20)
    y = np.linspace(-5, 5, 20)
    X, Y = np.meshgrid(x, y)
    
    # 2. Parámetros
    k = 8.99e9
    
    # 3. Cálculo del campo
    dx = X - x_c
    dy = Y - y_c
    r = np.sqrt(dx**2 + dy**2)
    
    # Evitar divisiones por cero cerca de la carga
    r_safe = np.where(r < 0.2, 0.2, r)
    
    Ex = k * q_val * dx / r_safe**3
    Ey = k * q_val * dy / r_safe**3
    
    # Intensidad para el color
    E_mag = np.sqrt(Ex**2 + Ey**2)
    
    # 4. Graficar
    plt.figure(figsize=(8, 7))
    
    # Dibujamos las flechas (normalizadas para que no se amontonen, pero con color por intensidad)
    qvr = plt.quiver(X, Y, Ex/E_mag, Ey/E_mag, np.log10(E_mag), 
                     cmap='plasma', pivot='middle', alpha=0.8)
    
    # Dibujar la carga fuente
    color_c = 'red' if q_val > 0 else 'blue'
    plt.scatter([x_c], [y_c], c=color_c, s=300, edgecolors='black', linewidths=2, zorder=5)
    plt.text(x_c + 0.3, y_c + 0.3, f'q = {q_val*1e6:.1f} µC', fontsize=10, 
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    plt.title(f'Campo Eléctrico Interactivo: Carga en ({x_c}, {y_c})')
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.grid(True, alpha=0.2)
    plt.colorbar(qvr, label='Intensidad del campo (Escala log$_{10}$)')
    plt.show()

# Configuración de los controles interactivos
interact(interactuar_campo, 
         q_val = RadioButtons(options=[1e-6, -1e-6], value=1e-6, description='Carga:'),
         x_c = FloatSlider(min=-4, max=4, step=0.1, value=0, description='Posición X:'),
         y_c = FloatSlider(min=-4, max=4, step=0.1, value=0, description='Posición Y:'))

interactive(children=(RadioButtons(description='Carga:', options=(1e-06, -1e-06), value=1e-06), FloatSlider(va…

<function __main__.interactuar_campo(q_val, x_c, y_c)>

In [2]:
from IPython.display import display, HTML

phet_iframe = """
<div style="width: 100%; text-align: center; margin-bottom: 20px;">
    <h3 style="margin-bottom: 10px;">Laboratorio Virtual: Cargas y Campos</h3>
    <iframe src="https://phet.colorado.edu/sims/html/charges-and-fields/latest/charges-and-fields_es.html"
        width="100%"
        height="500"
        scrolling="no"
        allowfullscreen>
    </iframe>
</div>
"""

display(HTML(phet_iframe))

In [3]:
import numpy as np
import plotly.graph_objects as go

def generar_campo_estatico_plotly(q_val, x_c, y_c):
    x = np.linspace(-5, 5, 20)
    y = np.linspace(-5, 5, 20)
    X, Y = np.meshgrid(x, y)
    
    k = 8.99e9
    dx = X - x_c
    dy = Y - y_c
    r = np.sqrt(dx**2 + dy**2)
    r_safe = np.where(r < 0.2, 0.2, r)
    
    Ex = k * q_val * dx / r_safe**3
    Ey = k * q_val * dy / r_safe**3
    E_mag = np.sqrt(Ex**2 + Ey**2)
    
    # Normalizamos para las flechas de Plotly
    Ex_n = Ex / E_mag
    Ey_n = Ey / E_mag

    # Creamos el gráfico de vectores
    fig = go.Figure(data=go.Cone(
        x=X.flatten(), y=Y.flatten(), z=np.zeros(X.size),
        u=Ex_n.flatten(), v=Ey_n.flatten(), w=np.zeros(X.size),
        sizemode="absolute", sizeref=0.3,
        colorscale='Plasma',
        colorbar=dict(title="Intensidad E")
    ))

    # Añadimos la carga
    fig.add_trace(go.Scatter(
        x=[x_c], y=[y_c],
        mode='markers+text',
        marker=dict(size=20, color='red' if q_val > 0 else 'blue'),
        text=[f"q={q_val*1e6}uC"],
        textposition="top center"
    ))

    fig.update_layout(title="Campo Eléctrico Interactivo (Plotly)",
                      xaxis_range=[-5, 5], yaxis_range=[-5, 5],
                      width=700, height=600)
    fig.show()

# Nota: Para que sea interactivo en la web con sliders, 
# Plotly requiere definir "updatemenus" o "sliders" internos de JS.

In [4]:
import numpy as np
import plotly.graph_objects as go

# 1. Generar datos base
x = np.linspace(-5, 5, 15)
y = np.linspace(-5, 5, 15)
X, Y = np.meshgrid(x, y)
k = 8.99e9

# 2. Crear la figura base
fig = go.Figure()

# 3. Crear "Frames" para la animación/slider (moviendo la carga en X)
pasos_x = np.linspace(-4, 4, 20)
for xc in pasos_x:
    dx = X - xc
    dy = Y - 0  # Carga fija en Y=0 para este ejemplo
    r = np.sqrt(dx**2 + dy**2)
    r_safe = np.where(r < 0.2, 0.2, r)
    Ex = k * 1e-6 * dx / r_safe**3
    Ey = k * 1e-6 * dy / r_safe**3
    
    # Añadir flechas y la bolita de la carga
    fig.add_trace(go.Quiver(
        x=X, y=Y, u=Ex, v=Ey, 
        scale=0.1, arrow_scale=.4, name=f"pos_{xc}", visible=False
    ))

# Hacer visible el primer frame
fig.data[0].visible = True

# 4. Crear el Slider de Plotly (JavaScript puro)
steps = []
for i in range(len(pasos_x)):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(fig.data)}],
        label=f"{pasos_x[i]:.1f}"
    )
    step["args"][0]["visible"][i] = True  
    steps.append(step)

sliders = [dict(active=0, currentvalue={"prefix": "Posición X: "}, pad={"t": 50}, steps=steps)]

fig.update_layout(sliders=sliders, xaxis_range=[-5,5], yaxis_range=[-5,5], height=600)
fig.show()

AttributeError: module 'plotly.graph_objects' has no attribute 'Quiver'